# Milestone 3 — Notebook 2: Hyperparameter search and MLflow (Lab 8)


## 1. Install MLflow and reconnect to Milestone 2 data

In [21]:
!pip -q install mlflow

In [22]:
# Google Cloud Storage settings
BUCKET_NAME = "conor-weekes-data-20084449"
FOLDER = f"gs://{BUCKET_NAME}/milestone2/processed"
RANDOM_SEED = 42

!pip -q install gcsfs
!pip -q install mlflow

from google.colab import auth
auth.authenticate_user()

In [23]:
import pandas as pd
import numpy as np

# Same processed CSV files you from Milestone 2.
train = pd.read_csv(f"{FOLDER}/train_processed.csv")
dev = pd.read_csv(f"{FOLDER}/dev_processed.csv")
test = pd.read_csv(f"{FOLDER}/test_processed.csv")

features = ["fund_return_2017", "fund_return_2018", "fund_return_2019"]
target = "fund_return_2021_q1"

X_train = train[features].to_numpy(dtype="float32")
y_train = train[target].to_numpy(dtype="float32")
X_dev = dev[features].to_numpy(dtype="float32")
y_dev = dev[target].to_numpy(dtype="float32")

print("Training:", train.shape, "Development:", dev.shape, "Test:", test.shape)

Training: (18107, 5) Development: (2265, 5) Test: (2256, 5)


In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Set up MLflow


In [25]:
import mlflow
import os
import time
import json
import joblib
import platform
import hashlib
import sklearn
from datetime import datetime, timezone
from pathlib import Path

results_folder = Path("/content/m3_results" if Path("/content").exists() else "m3_results")
results_folder.mkdir(exist_ok=True)
tracking_folder = results_folder / "mlruns"
tracking_folder.mkdir(exist_ok=True)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri(tracking_folder.resolve().as_uri())
mlflow.set_experiment("Mutual_Funds_M3")
print("MLflow:", mlflow.get_tracking_uri())
print("CPU cores:", os.cpu_count())


MLflow: file:///content/m3_results/mlruns
CPU cores: 2


## 3. Define the same sort of grid search as Lab 8

The development MAE is compared. The baseline is included as additional model type.

In [26]:

import itertools
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.dummy import DummyRegressor

HIDDEN_SIZES = [8, 16, 32]
LEARNING_RATES = [0.001, 0.003, 0.01]
EPOCHS = 15
BATCH_SIZE = 256
DEVICE = torch.device("cpu")

class FundMLP(nn.Module):
    def __init__(self, hidden_size):
        super(FundMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(3, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.network(x)

xtr = torch.FloatTensor(np.array(X_train).astype(np.float32)).to(DEVICE)
ytr = torch.FloatTensor(np.array(y_train).astype(np.float32)).unsqueeze(1).to(DEVICE)
xdev = torch.FloatTensor(np.array(X_dev).astype(np.float32)).to(DEVICE)


## 4. Run experiments and record everything in MLflow

Each run logs the hyperparameters, development metrics, training duration, and learning curve. The dummy model use the same evaluation metric so the results can be compared.

In [31]:


records = []
best_record = None
best_model = None

# Run the DummyRegressor baseline.
model = DummyRegressor(strategy="mean")

with mlflow.start_run(run_name="Dummy_baseline") as run:
    mlflow.log_param("model_type", "Dummy")
    mlflow.log_param("seed", RANDOM_SEED)

    start = time.time()
    model.fit(X_train, y_train)
    seconds = time.time() - start

    pred = model.predict(X_dev)

    mae = mean_absolute_error(y_dev, pred)
    rmse = np.sqrt(mean_squared_error(y_dev, pred))
    r2 = r2_score(y_dev, pred)

    mlflow.log_metric("dev_mae", mae)
    mlflow.log_metric("dev_rmse", rmse)
    mlflow.log_metric("dev_r2", r2)
    mlflow.log_metric("train_seconds", seconds)

    record = {
        "model_type": "Dummy",
        "hidden_size": None,
        "learning_rate": None,
        "dev_mae": mae,
        "dev_rmse": rmse,
        "dev_r2": r2,
        "train_seconds": seconds,
        "run_id": run.info.run_id
    }

    records.append(record)
    best_record = record
    best_model = model

print("Dummy MAE:", round(mae * 100, 3), "percentage points")


Dummy MAE: 4.379 percentage points


In [32]:

import torch.optim as optim
import mlflow.pytorch

def train_and_eval_pytorch_mlflow(X_tr, y_tr, X_de, y_de, device, hidden_size, lr, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        start_time = time.time()

        # Convert data to PyTorch Tensors and cast to float32
        X_tr_t = torch.FloatTensor(np.array(X_tr).astype(np.float32)).to(device)
        y_tr_t = torch.FloatTensor(np.array(y_tr).astype(np.float32)).unsqueeze(1).to(device)
        X_de_t = torch.FloatTensor(np.array(X_de).astype(np.float32)).to(device)
        y_de_t = torch.FloatTensor(np.array(y_de).astype(np.float32)).unsqueeze(1).to(device)

        dataset = TensorDataset(X_tr_t, y_tr_t)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

        model = FundMLP(hidden_size).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Log hyperparameters to MLflow
        mlflow.log_param("input_size", X_tr.shape[1])
        mlflow.log_param("hidden_size", hidden_size)
        mlflow.log_param("epochs", EPOCHS)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("device", str(device))

        loss_history = []

        # Training loop
        for epoch in range(EPOCHS):
            model.train()
            epoch_loss = 0.0

            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * batch_x.size(0)

            avg_epoch_loss = epoch_loss / len(X_tr)
            loss_history.append(avg_epoch_loss)
            mlflow.log_metric("loss", avg_epoch_loss, step=epoch)

            model.eval()
            with torch.no_grad():
                dev_loss = criterion(model(X_de_t), y_de_t).item()

            mlflow.log_metric("dev_loss", dev_loss, step=epoch)

        # Evaluation
        model.eval()
        with torch.no_grad():
            predictions = model(X_de_t).cpu().numpy().flatten()

        mae = mean_absolute_error(y_de, predictions)
        rmse = np.sqrt(mean_squared_error(y_de, predictions))
        r2 = r2_score(y_de, predictions)

        duration = time.time() - start_time

        # Log final metrics and model
        mlflow.log_metric("dev_mae", mae)
        mlflow.log_metric("dev_rmse", rmse)
        mlflow.log_metric("dev_r2", r2)
        mlflow.log_metric("train_seconds", duration)

        mlflow.pytorch.log_model(model, "model", serialization_format="pickle")

        print(f"[{run_name}] MAE: {mae * 100:.3f} percentage points")

    return mae, rmse, r2, loss_history, model, duration, run.info.run_id


# Define Grid Search Parameters
hidden_sizes = [8, 16, 32]
learning_rates = [0.001, 0.003, 0.01]

print("=== Starting Grid Search with MLflow Logging ===")

for hs, lr in itertools.product(hidden_sizes, learning_rates):

    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_title = f"Fund_MLP_HS_{hs}_LR_{lr}_{timestamp}"

    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    mae, rmse, r2, loss_hist, model, duration, run_id = train_and_eval_pytorch_mlflow(
        X_train, y_train, X_dev, y_dev, DEVICE, hs, lr, run_name=run_title
    )

    record = {
        "model_type": "PyTorch",
        "hidden_size": hs,
        "learning_rate": lr,
        "dev_mae": mae,
        "dev_rmse": rmse,
        "dev_r2": r2,
        "train_seconds": duration,
        "run_id": run_id
    }

    records.append(record)

    if mae < best_record["dev_mae"]:
        best_record = record
        best_model = model

results_df = pd.DataFrame(records).sort_values("dev_mae")
display(results_df)

print("\n=== Best Hyperparameter Set Found ===")
print("Best Model:", best_record)


=== Starting Grid Search with MLflow Logging ===


2026/11/20 16:52:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 16:52:45 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/20 16:52:45 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.14.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/20 16:52:50 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cpu) contains a local version label (+cpu

[Fund_MLP_HS_8_LR_0.001_20261120-165238] MAE: 3.116 percentage points


2026/11/20 16:52:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 16:52:56 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/20 16:52:56 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.14.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/20 16:53:02 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cpu) contains a local version label (+cpu

[Fund_MLP_HS_8_LR_0.003_20261120-165250] MAE: 3.002 percentage points


2026/11/20 16:53:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 16:53:08 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/20 16:53:08 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.14.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/20 16:53:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cpu) contains a local version label (+cpu

... (11 outputs left out)

2026/11/20 16:54:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 16:54:18 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/20 16:54:18 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.14.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/20 16:54:23 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cpu) contains a local version label (+cpu

[Fund_MLP_HS_32_LR_0.01_20261120-165413] MAE: 2.867 percentage points


,model_type,hidden_size,learning_rate,dev_mae,dev_rmse,dev_r2,train_seconds,run_id
6,PyTorch,16.0,0.010,0.027599,0.040081,0.534536,5.670072,3a337b8d0e48439fb2fed195105d64c2
9,PyTorch,32.0,0.010,0.028669,0.041011,0.512681,4.651172,2233e8e181cb48a18193174642514869
8,PyTorch,32.0,0.003,0.029390,0.042257,0.482640,4.217471,d908ce9791464282999b9b3fa6f3a8e0
3,PyTorch,8.0,0.010,0.029432,0.041747,0.495054,6.395485,60e635af37894b1fba430ef5e7c84fa7
5,PyTorch,16.0,0.003,0.030008,0.042497,0.476733,3.995317,964f43ef23fa475b9ba2522968ddd910
2,PyTorch,8.0,0.003,0.030023,0.043357,0.455349,6.032196,557cbd3b685b4f839bd80ef844187e2b
1,PyTorch,8.0,0.001,0.031157,0.045247,0.406833,7.257889,e3e3d5103ebc45b7a796f75ee707c9f3
7,PyTorch,32.0,0.001,0.031844,0.045803,0.392152,4.182393,c2d739f48374420cbba3cba2bdc5b5e9
4,PyTorch,16.0,0.001,0.033620,0.046727,0.367389,6.264177,3ee8207d516a4cc6981f9f70abdcc881
0,Dummy,NaN,NaN,0.043785,0.058758,-0.000326,0.001336,0d85c9aa9bcf4da0adaa407f1191f7c6



=== Best Hyperparameter Set Found ===
Best Model: {'model_type': 'PyTorch', 'hidden_size': 16, 'learning_rate': 0.01, 'dev_mae': 0.027599377557635307, 'dev_rmse': np.float64(0.040081259252175226), 'dev_r2': 0.5345362424850464, 'train_seconds': 5.670071840286255, 'run_id': '3a337b8d0e48439fb2fed195105d64c2'}


## 5. Save the chosen model and the experiment log


In [33]:

# Save experiment results.
results_df.to_csv(results_folder / "experiments.csv", index=False)

# Save the selected model.
torch.save(best_model.state_dict(), results_folder / "selected_model.pt")

# Save information needed to load the model in Notebook 3.
selection = {
    "best_run": {
        "model_type": best_record["model_type"],
        "hidden_size": int(best_record["hidden_size"]),
        "learning_rate": float(best_record["learning_rate"]),
        "dev_mae": float(best_record["dev_mae"]),
        "train_seconds": float(best_record["train_seconds"]),
        "run_id": best_record["run_id"]
    },
    "features": features,
    "target": target
}

(results_folder / "selected_model.json").write_text(
    json.dumps(selection, indent=2)
)

print("Saved experiments and selected model in", results_folder)


Saved experiments and selected model in /content/m3_results


## 6. Keep your results after Colab disconnects

Upload the experiment results, selected model and configuration to GCS so Notebook 3 can access them.

In [34]:
from google.cloud import storage

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

files_to_upload = [
    "experiments.csv",
    "selected_model.pt",
    "selected_model.json"
]

for filename in files_to_upload:
    local_path = results_folder / filename
    cloud_path = f"milestone3/{filename}"

    blob = bucket.blob(cloud_path)
    blob.upload_from_filename(str(local_path))

    print(f"Uploaded to gs://{BUCKET_NAME}/{cloud_path}")


Uploaded to gs://conor-weekes-data-20084449/milestone3/experiments.csv
Uploaded to gs://conor-weekes-data-20084449/milestone3/selected_model.pt
Uploaded to gs://conor-weekes-data-20084449/milestone3/selected_model.json
